# 1. Data Acquisition and Preprocessing

Builds both datasets from source and documents every decision taken about them.

## What is real and what is simulated

**Real field data.** The 23 service towns, their road distances from the Hohoe
ECG base, and their travel times. Collected from the Hohoe Operations Department.
This is the project's primary asset.

**Simulated.** The 15 faults. They are drawn with `random.choices` against an
assumed fault-type mix and an assumed 30/70 priority split, neither of which has
a published source.

Simulated demand against a real network is a standard and acceptable study
design. **Undisclosed, it is a research-integrity problem** — so this must be
stated in the paper's *abstract*, not only in Methods.

In [1]:
import pathlib, sys
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src.config import SUSPECT_TOWNS, ZONE_THRESHOLD_KM, ZONE_THRESHOLD_SENSITIVITY
from src.data import classify_zone, describe_instance, load_towns
from src.generate import (FAULT_TYPES, PRIORITY_WEIGHTS, PUBLISHED_SEED,
                          TYPE_WEIGHTS, generate_faults, reproduces_published)
from src.speed_model import fit, outlier_report, power_law, threshold_placebo

pd.set_option("display.width", 120)
print(f"zone threshold in use: {ZONE_THRESHOLD_KM} km")

zone threshold in use: 20.0 km


## 1.1 The service-town network

23 towns with field-measured distance and travel time from the base.

In [2]:
towns = load_towns()
towns[["service_town", "distance_in_km", "travel_time_in_hours",
       "zone", "implied_speed_kmh"]].sort_values("distance_in_km")

,service_town,distance_in_km,travel_time_in_hours,zone,implied_speed_kmh
12,Fodome,0.65,0.650,Near,1.000000
17,Gbledi,1.30,0.050,Near,26.000000
0,Gbi - Kledzo,4.90,0.267,Near,18.352060
1,Gbi - Wegbe,6.20,0.333,Near,18.618619
19,Santrokofi,7.50,0.333,Near,22.522523
21,Lolobi,10.10,0.350,Near,28.857143
3,Gbi - Avege,10.20,0.467,Near,21.841542
20,Akpafu,11.90,0.417,Near,28.537170
5,Ve- Kobenu,13.70,0.583,Near,23.499142
4,Ve-Dator,14.30,0.600,Near,23.833333


## 1.2 Zone classification — a decision taken before any result was generated

The original code set the Near/Far boundary at `ceil(max(distance))/2` = 19.5 km.
That is a **function of the farthest town in the dataset**: adding one distant
town silently reclassifies others. The README stated 20 km. The two disagree on
**Liati, at exactly 20.0 km**.

The threshold is now an exogenous constant in `src/config.py`. Changing it moves
every equity constant in the paper, which is why it was fixed first.

In [3]:
rows = []
for thr in ZONE_THRESHOLD_SENSITIVITY:
    z = towns["distance_in_km"].apply(classify_zone, threshold_km=thr)
    rows.append({"threshold_km": thr,
                 "n_near": int((z == "Near").sum()),
                 "n_far": int((z == "Far").sum()),
                 "Liati": classify_zone(20.0, thr)})
print(pd.DataFrame(rows).to_string(index=False))
print(f"\nin use: {ZONE_THRESHOLD_KM} km (src.config.ZONE_THRESHOLD_KM)")

 threshold_km  n_near  n_far Liati
         19.5      14      9   Far
         20.0      15      8  Near
         25.0      18      5  Near

in use: 20.0 km (src.config.ZONE_THRESHOLD_KM)


## 1.3 The travel-time model

The first audit flagged the implied road speeds — distance ÷ travel time —
as spanning 1.0 to 47.2 km/h and called it a data-quality problem.

**It is mostly not.** Regressing travel time on distance recovers a fixed
overhead plus a roughly constant cruise speed. Dividing a *fixed* overhead by a
*short* distance is what produces an apparently slow town.

This turns the dataset's most reviewer-visible oddity into a finding the paper
can report, rather than an assumption it has to defend.

In [4]:
sf = fit(towns)
print(sf.summary())
print(f"\n  intercept {sf.intercept_h:.4f} h = {sf.overhead_min:.1f} min"
      f"  (se {sf.intercept_se:.4f}, p = {sf.intercept_p:.2e})")
print(f"  slope     {sf.slope_h_per_km:.5f} h/km -> cruise {sf.cruise_kmh:.2f} km/h"
      f"  (p = {sf.slope_p:.2e})")
print(f"  excluded from the fit: {sf.excluded}  (see section 1.4)")

t = 0.1595 + 0.02059 d   (n=22, R2=0.8957, LOOCV Q2=0.8743)
  fixed overhead 9.6 min, cruise 48.57 km/h, LOOCV MAE 3.72 min

  intercept 0.1595 h = 9.6 min  (se 0.0334, p = 1.17e-04)
  slope     0.02059 h/km -> cruise 48.57 km/h  (p = 2.82e-11)
  excluded from the fit: ['Fodome']  (see section 1.4)


In [5]:
# An independent route to the same conclusion. Constant speed would mean k = 1.
pl = power_law(towns)
print(f"power law  t = c * d^k")
print(f"  k = {pl['k']:.4f}   95% CI [{pl['ci95'][0]:.4f}, {pl['ci95'][1]:.4f}]"
      f"   R2 = {pl['r2']:.4f}")
print(f"  H0 (k = 1, constant speed): t = {pl['t_vs_constant_speed']:.3f},"
      f" p = {pl['p_vs_constant_speed']:.2e}")
print("\n  k < 1 means effective speed RISES with distance -- which is exactly")
print("  what a fixed per-trip overhead produces.")

power law  t = c * d^k
  k = 0.7415   95% CI [0.6272, 0.8558]   R2 = 0.9015
  H0 (k = 1, constant speed): t = -4.718, p = 1.32e-04

  k < 1 means effective speed RISES with distance -- which is exactly
  what a fixed per-trip overhead produces.


### Is there a genuine Near/Far break in travel time?

A Near/Far step dummy added to the linear model looks significant (p ≈ 0.039).
Before that goes anywhere near the paper it has to survive a placebo test:
**if a dummy at many thresholds is significant, the one we happened to pick is
not a finding.**

In [6]:
pb = threshold_placebo(towns)
at20 = next(r for r in pb["tests"] if r["threshold_km"] == 20)
print(f"candidate thresholds tested      : {pb['n_tests']}")
print(f"significant at p < 0.05          : {pb['n_significant_uncorrected']}")
print(f"best threshold                   : {pb['best']['threshold_km']:.0f} km"
      f" (p = {pb['best']['p']:.4f})")
print(f"Bonferroni alpha                 : {pb['bonferroni_alpha']:.5f}")
print(f"surviving correction             : {pb['surviving_bonferroni']}")
print(f"\nAt the {ZONE_THRESHOLD_KM:.0f} km threshold the optimisation model actually uses:")
print(f"  F = {at20['F']:.2f}, p = {at20['p']:.4f}"
      f"  -> {'significant' if at20['p'] < 0.05 else 'NOT significant'}")
print("\nConclusion: no separate zone term is warranted. The step dummy is a")
print("competing parameterisation of the same fixed overhead, the best-fitting")
print("threshold (16 km) is not the one the model uses, and reporting it would")
print("be a specification search over 29 candidates.")

candidate thresholds tested      : 29
significant at p < 0.05          : 5
best threshold                   : 16 km (p = 0.0007)
Bonferroni alpha                 : 0.00172
surviving correction             : [16.0]

At the 20 km threshold the optimisation model actually uses:
  F = 4.11, p = 0.0568  -> NOT significant

Conclusion: no separate zone term is warranted. The step dummy is a
competing parameterisation of the same fixed overhead, the best-fitting
threshold (16 km) is not the one the model uses, and reporting it would
be a specification search over 29 candidates.


## 1.4 Data quality — documented, never silently patched

A reviewer who downloads the CSV, finds an anomaly, and sees no mention of it
will assume the worst. Everything known is in `docs/data_quality.md`; the
machine-checkable part is in `src/config.SUSPECT_TOWNS`.

In [7]:
for o in outlier_report(towns, sf):
    print(f"{o['town']}")
    print(f"  recorded                     : {o['recorded_km']} km in "
          f"{o['recorded_hours']} h = {o['implied_speed_kmh']:.2f} km/h")
    print(f"  model predicts for that distance: {o['model_predicts_hours'] * 60:.1f} min")
    print(f"  distance implied by that TIME   : {o['distance_implied_by_time_km']:.1f} km")
    print(f"  residual                        : {o['residual_sd']:.1f} sd")
    print(f"  decision                        : {o['note']}")
print("\nThe value is RETAINED unchanged and excluded from the fit. It is not")
print("corrected because the true value is unknown -- that needs the original")
print("measurement source. No fault in the published instance occurs here.")

Fodome
  recorded                     : 0.65 km in 0.65 h = 1.00 km/h
  model predicts for that distance: 10.4 min
  distance implied by that TIME   : 23.8 km
  residual                        : 6.5 sd
  decision                        : 0.65 km in 0.65 h implies 1.0 km/h. Near-certain transcription error; likely 6.5 km. Excluded from the speed-model calibration. No fault in the published instance occurs here.

The value is RETAINED unchanged and excluded from the fit. It is not
corrected because the true value is unknown -- that needs the original
measurement source. No fault in the published instance occurs here.


## 1.5 Fault simulation

Reproduces `dataset/ecg_faults_dataset.csv` exactly at seed 60. The assertion
below is what stops this module and the committed dataset from silently
diverging.

In [8]:
print("ASSUMED fault-type mix (no published source -- state as an assumption):")
for (name, repair), wgt in zip(FAULT_TYPES.items(), TYPE_WEIGHTS):
    print(f"  {wgt:>5.0%}  {name:<32} repair {repair} h")
print(f"\nASSUMED priority split: {PRIORITY_WEIGHTS[0]:.0%} High / {PRIORITY_WEIGHTS[1]:.0%} Normal")
print("\nObtaining ECG's historical fault log would replace both with measured")
print("frequencies and materially strengthen the paper.")

ok = reproduces_published(ROOT / "dataset" / "ecg_faults_dataset.csv")
assert ok, "generator and committed dataset have diverged -- stop and investigate"
print(f"\ngenerator reproduces the committed dataset at seed {PUBLISHED_SEED}: {ok}")

ASSUMED fault-type mix (no published source -- state as an assumption):
    15%  Transformer installation         repair 4.0 h
    25%  Transformer maintenance          repair 0.75 h
    20%  Cable joining and termination    repair 0.33 h
    15%  Network line extension           repair 2.0 h
    10%  Pole replacement                 repair 1.0 h
    15%  Vegetation control               repair 4.0 h

ASSUMED priority split: 30% High / 70% Normal

Obtaining ECG's historical fault log would replace both with measured
frequencies and materially strengthen the paper.

generator reproduces the committed dataset at seed 60: True


In [9]:
faults = generate_faults(15, seed=PUBLISHED_SEED)
d = describe_instance(faults)
print("| Quantity | Value |")
print("|---|---|")
for k, v in [("Faults", d["n_faults"]), ("Distinct towns", d["distinct_towns"]),
             ("Near-zone faults", d["n_near"]), ("Far-zone faults", d["n_far"]),
             ("Mean one-way travel, Near", f"{d['mean_travel_near']:.4f} h"),
             ("Mean one-way travel, Far", f"{d['mean_travel_far']:.4f} h"),
             ("Total repair time", f"{d['total_repair']:.3f} h"),
             ("High-priority faults", d["n_high_priority"]),
             ("...of which in Far towns", d["n_high_priority_far"])]:
    print(f"| {k} | {v} |")
faults

| Quantity | Value |
|---|---|
| Faults | 15 |
| Distinct towns | 12 |
| Near-zone faults | 9 |
| Far-zone faults | 6 |
| Mean one-way travel, Near | 0.4779 h |
| Mean one-way travel, Far | 0.7807 h |
| Total repair time | 20.070 h |
| High-priority faults | 5 |
| ...of which in Far towns | 4 |


,Fault_ID,Town,Zone,Distance_km,Travel_time_hours,Fault_type,Repair_time_hours,Priority
0,F1,Afadzo South,Far,38.4,1.017,Pole replacement,1.00,High
1,F2,Liati,Near,20.0,0.550,Transformer maintenance,0.75,Normal
2,F3,Golokwati,Far,21.3,0.500,Transformer maintenance,0.75,High
3,F4,Agome yo,Near,15.9,0.517,Network line extension,2.00,Normal
4,F5,Logba,Far,38.3,1.000,Network line extension,2.00,Normal
5,F6,Afadzo South,Far,38.4,1.017,Cable joining and termination,0.33,High
6,F7,Akpafu,Near,11.9,0.417,Cable joining and termination,0.33,Normal
7,F8,Agome yo,Near,15.9,0.517,Cable joining and termination,0.33,High
8,F9,Santrokofi,Near,7.5,0.333,Network line extension,2.00,Normal
9,F10,Wli,Far,22.8,0.483,Transformer installation,4.00,High


> **Note for the equity analysis.** Four of the five high-priority faults are in
> Far towns. Weighting priority therefore moves the Far/Near ratio with no equity
> mechanism doing any work, so every equity experiment runs at `w_High = 1`.

## 1.6 Write the datasets

These two lines were **commented out** in the original notebook, so running it
did not regenerate the committed CSVs and a reviewer could not verify their
provenance.

In [10]:
towns.drop(columns=["implied_speed_kmh", "data_quality_flag"]).to_csv(
    ROOT / "dataset" / "ecg_towns_dataset.csv", index=False)
faults.to_csv(ROOT / "dataset" / "ecg_faults_dataset.csv", index=False)
print("written:")
print("  dataset/ecg_towns_dataset.csv")
print("  dataset/ecg_faults_dataset.csv")

written:
  dataset/ecg_towns_dataset.csv
  dataset/ecg_faults_dataset.csv
